# 04 - Model Optimization & Robust Backtesting

## Time Series Forecasting with Intelligent Cross-Validation

Notebook pentru optimizare și evaluare robustă a modelelor de prognoză pe serii temporale zilnice.
Implementează backtesting cu expanding/rolling windows, multiple forecast horizons, și evaluare detaliată.

**Modele testrate:**
- Baseline: Naive, Seasonal Naive, Moving Average
- ARIMA/SARIMA (cu parametri optimizați)
- LSTM (deep learning)
- XGBoost (cu feature engineering)
- Prophet (Facebook)
- Exponential Smoothing

## 1. IMPORTS & CONFIGURATION

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Time Series
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.seasonal import seasonal_decompose, STL
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from pmdarima import auto_arima

# ML & Optimization
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import (
    mean_absolute_error, 
    mean_squared_error, 
    mean_absolute_percentage_error,
    r2_score
)
import xgboost as xgb

# Deep Learning
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

# Prophet
from prophet import Prophet
import logging
logging.getLogger('prophet').setLevel(logging.WARNING)

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("✓ All libraries imported successfully")

## 2. DATA LOADING & PREPARATION

In [ ]:
# ============================================================================
# CONFIGURATION PARAMETERS
# ============================================================================
CONFIG = {
    'csv_file': '../data_raw/date_24_art1_loreal.csv',
    'target_column': 'CANTITATE',
    'date_column': 'DATA',
    'article_id': None,  # None = toate articolele; dacă set: filtrează la ID specific
    'frequency': 'D',  # Daily data
    'forecast_horizons': [7, 14, 30],  # Test forecasts for 7, 14, 30 days ahead
    'min_train_size': 60,  # Minimum training observations
    'n_folds': 5,  # Number of backtesting folds
}

print(f"Configuration:")
print(f"  CSV: {CONFIG['csv_file']}")
print(f"  Target: {CONFIG['target_column']}")
print(f"  Forecast horizons: {CONFIG['forecast_horizons']} days")
print(f"  Min train size: {CONFIG['min_train_size']} observations")
print(f"  CV folds: {CONFIG['n_folds']}\n")

In [ ]:
def prepare_series(csv_file, date_col, target_col, article_id=None, freq='D'):
    """
    Load and prepare time series data.
    
    Returns:
        series (pd.Series): Time series indexed by date
        df_raw (pd.DataFrame): Raw data for reference
    """
    df = pd.read_csv(csv_file)
    
    # Convert date
    df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
    df = df.dropna(subset=[date_col])
    
    # Filter by article if needed
    if article_id is not None and 'ID_ARTICOL' in df.columns:
        df = df[df['ID_ARTICOL'] == article_id].copy()
        print(f"Filtered to article {article_id}: {len(df)} observations")
    
    # Sort and aggregate by date (in case multiple entries per day)
    df_agg = df.groupby(date_col)[target_col].sum().reset_index()
    df_agg = df_agg.sort_values(date_col)
    
    # Create time series with daily frequency
    series = df_agg.set_index(date_col)[target_col].astype(float)
    series = series.asfreq(freq)
    
    # Fill missing values
    series = series.fillna(method='ffill').fillna(0)
    
    # Remove trailing zeros (often indicate data cutoff)
    while len(series) > 0 and series.iloc[-1] == 0:
        series = series[:-1]
    
    print(f"✓ Series prepared: {len(series)} observations")
    print(f"  Date range: {series.index.min().date()} to {series.index.max().date()}")
    print(f"  Mean: {series.mean():.2f}, Std: {series.std():.2f}")
    print(f"  Min: {series.min():.2f}, Max: {series.max():.2f}\n")
    
    return series, df_agg

# Load data
series, df_raw = prepare_series(
    CONFIG['csv_file'],
    CONFIG['date_column'],
    CONFIG['target_column'],
    article_id=CONFIG['article_id'],
    freq=CONFIG['frequency']
)

# Quick visualization
fig = go.Figure()
fig.add_trace(go.Scatter(x=series.index, y=series.values, mode='lines', name='Target'))
fig.update_layout(
    title='Time Series Overview',
    xaxis_title='Date',
    yaxis_title=CONFIG['target_column'],
    height=400,
    width=1200,
    hovermode='x unified'
)
fig.show()

## 3. INTELLIGENT CROSS-VALIDATION STRATEGY

In [ ]:
def determine_seasonality(series, freq='D'):
    """Detect seasonal period (e.g., 7 for daily = weekly)."""
    if freq == 'D':
        return 7  # Weekly seasonality for daily data
    elif freq == 'W':
        return 52  # Yearly seasonality for weekly data
    elif freq == 'M':
        return 12  # Yearly seasonality for monthly data
    else:
        return 7  # Default

def generate_cv_splits(series, n_folds=5, min_train_size=60, horizon = 30):
    """
    Generate expanding window CV splits for time series.
    
    Returns:
        list of tuples: (train_indices, test_indices, fold_number)
    """
    n = len(series)
    
    # Adjust number of folds based on series length
    max_folds = max(1, (n - min_train_size) // (horizon + 10))
    n_folds = min(n_folds, max_folds)
    
    if n_folds < 1:
        raise ValueError(f"Series too short ({n} obs) for {min_train_size} min train + {horizon} test")
    
    splits = []
    test_size_per_fold = (n - min_train_size) // n_folds
    
    for fold in range(n_folds):
        # Expanding window: train grows, test is fixed size
        train_end = min_train_size + fold * test_size_per_fold
        test_end = min(train_end + horizon, n)
        
        train_idx = np.arange(0, train_end)
        test_idx = np.arange(train_end, test_end)
        
        if len(train_idx) >= min_train_size and len(test_idx) > 0:
            splits.append((train_idx, test_idx, fold + 1))
    
    print(f"Generated {len(splits)} CV folds (horizon={horizon} days):")
    for train_idx, test_idx, fold in splits:
        print(f"  Fold {fold}: train={len(train_idx)}, test={len(test_idx)}")
    
    return splits

# Generate CV splits for each horizon
cv_splits_by_horizon = {}
for horizon in CONFIG['forecast_horizons']:
    cv_splits_by_horizon[horizon] = generate_cv_splits(
        series,
        n_folds=CONFIG['n_folds'],
        min_train_size=CONFIG['min_train_size'],
        horizon=horizon
    )
    print()

## 4. METRICS & EVALUATION FUNCTIONS

In [ ]:
def calculate_metrics(y_true, y_pred):
    """Calculate comprehensive evaluation metrics."""
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    
    # Ensure non-negative for MAPE
    y_pred = np.maximum(y_pred, 0)
    
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)
    
    # SMAPE (symmetric)
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    smape = np.mean(2 * np.abs(y_true - y_pred) / (denominator + 1e-10)) * 100
    
    # MASE (mean absolute scaled error)
    naive_error = np.mean(np.abs(np.diff(y_true)))
    mase = mae / (naive_error + 1e-10) if naive_error > 0 else 0
    
    r2 = r2_score(y_true, y_pred) if len(y_true) > 0 else 0
    
    return {
        'MAE': mae,
        'RMSE': rmse,
        'MAPE': mape,
        'SMAPE': smape,
        'MASE': mase,
        'R2': r2
    }

print("✓ Metrics functions defined")

## 5. BASELINE MODELS

In [ ]:
def forecast_naive(series_train, horizon):
    """Naive forecast: repeat last value."""
    return np.full(horizon, series_train.iloc[-1])

def forecast_seasonal_naive(series_train, horizon, season=7):
    """Seasonal naive: use value from same season."""
    n = len(series_train)
    pred = []
    for i in range(horizon):
        idx = max(0, n - season + (i % season))
        pred.append(series_train.iloc[idx])
    return np.array(pred)

def forecast_moving_average(series_train, horizon, window=7):
    """Moving average forecast."""
    ma_value = series_train.iloc[-window:].mean()
    return np.full(horizon, ma_value)

print("✓ Baseline models defined")

## 6. ARIMA/SARIMA OPTIMIZATION

In [ ]:
def fit_arima_sarima(series_train, horizon, use_seasonal=True, seasonal_period=7):
    """
    Fit ARIMA/SARIMA with auto-tuning.
    """
    try:
        # Auto ARIMA/SARIMA
        model = auto_arima(
            series_train,
            seasonal=use_seasonal,
            m=seasonal_period if use_seasonal else 1,
            max_p=4,
            max_q=4,
            max_d=2,
            max_P=2 if use_seasonal else 0,
            max_Q=2 if use_seasonal else 0,
            max_D=1 if use_seasonal else 0,
            stepwise=True,
            suppress_warnings=True,
            error_action='ignore',
            trace=False
        )
        
        # Forecast
        forecast = model.predict(n_periods=horizon)
        
        return np.array(forecast), model.order if not use_seasonal else (model.order, model.seasonal_order)
        
    except Exception as e:
        print(f"ARIMA/SARIMA failed: {str(e)[:50]}")
        # Fallback to naive
        return forecast_naive(series_train, horizon), None

print("✓ ARIMA/SARIMA function defined")

## 7. EXPONENTIAL SMOOTHING

In [ ]:
def fit_exponential_smoothing(series_train, horizon, seasonal_period=7):
    """
    Fit Exponential Smoothing (Holt-Winters).
    """
    try:
        # Additive ETS
        if len(series_train) >= 2 * seasonal_period:
            model = ExponentialSmoothing(
                series_train,
                seasonal_periods=seasonal_period,
                trend='add',
                seasonal='add',
                initialization_method='estimated'
            )
        else:
            # Fallback to simple exponential smoothing
            model = ExponentialSmoothing(
                series_train,
                trend='add',
                seasonal=None
            )
        
        fitted = model.fit(optimized=True)
        forecast = fitted.forecast(steps=horizon)
        
        return np.array(forecast), fitted
        
    except Exception as e:
        print(f"Exponential Smoothing failed: {str(e)[:50]}")
        return forecast_naive(series_train, horizon), None

print("✓ Exponential Smoothing function defined")

## 8. LSTM (DEEP LEARNING)

In [ ]:
def fit_lstm(series_train, horizon, lookback=14, epochs=50, batch_size=16):
    """
    Fit LSTM model for time series forecasting.
    """
    try:
        # Normalize
        scaler = MinMaxScaler(feature_range=(0, 1))
        data_scaled = scaler.fit_transform(series_train.values.reshape(-1, 1))
        
        # Create sequences
        X, y = [], []
        for i in range(len(data_scaled) - lookback):
            X.append(data_scaled[i:i+lookback, 0])
            y.append(data_scaled[i+lookback, 0])
        
        if len(X) < 10:  # Not enough data
            return forecast_naive(series_train, horizon), None, None
        
        X = np.array(X).reshape(-1, lookback, 1)
        y = np.array(y)
        
        # Build model
        model = Sequential([
            LSTM(64, activation='relu', return_sequences=True, input_shape=(lookback, 1)),
            Dropout(0.2),
            LSTM(32, activation='relu'),
            Dropout(0.2),
            Dense(16, activation='relu'),
            Dense(1)
        ])
        
        model.compile(optimizer=Adam(learning_rate=0.001), loss='mse')
        
        # Train
        es = EarlyStopping(monitor='loss', patience=5, restore_best_weights=True)
        model.fit(X, y, epochs=epochs, batch_size=batch_size, callbacks=[es], verbose=0)
        
        # Forecast
        last_sequence = data_scaled[-lookback:]
        predictions_scaled = []
        
        for _ in range(horizon):
            pred = model.predict(last_sequence.reshape(1, lookback, 1), verbose=0)[0, 0]
            predictions_scaled.append(pred)
            last_sequence = np.append(last_sequence[1:], pred)
        
        predictions = scaler.inverse_transform(np.array(predictions_scaled).reshape(-1, 1)).flatten()
        
        return np.array(predictions), model, scaler
        
    except Exception as e:
        print(f"LSTM failed: {str(e)[:50]}")
        return forecast_naive(series_train, horizon), None, None

print("✓ LSTM function defined")

## 9. XGBOOST (GRADIENT BOOSTING)

In [ ]:
def create_xgboost_features(series, lags=[1, 7, 14], windows=[7, 14]):
    """
    Create features for XGBoost from time series.
    """
    df = pd.DataFrame({'target': series.values}, index=series.index)
    
    # Lag features
    for lag in lags:
        df[f'lag_{lag}'] = df['target'].shift(lag)
    
    # Rolling mean & std
    for w in windows:
        df[f'rolling_mean_{w}'] = df['target'].shift(1).rolling(window=w).mean()
        df[f'rolling_std_{w}'] = df['target'].shift(1).rolling(window=w).std()
    
    # Time features
    df['dayofweek'] = df.index.dayofweek
    df['dayofyear'] = df.index.dayofyear
    df['month'] = df.index.month
    
    df = df.dropna()
    
    return df

def fit_xgboost(series_train, series_test, horizon):
    """
    Fit XGBoost with feature engineering.
    """
    try:
        # Combine for feature creation
        series_combined = pd.concat([series_train, series_test])
        df = create_xgboost_features(series_combined)
        
        # Split features and target
        n_train = len(series_train)
        X = df.drop('target', axis=1)
        y = df['target']
        
        X_train = X.iloc[:n_train].copy()
        y_train = y.iloc[:n_train].copy()
        
        # Train XGBoost
        model = xgb.XGBRegressor(
            n_estimators=300,
            max_depth=5,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            random_state=42
        )
        
        model.fit(X_train, y_train, verbose=0)
        
        # Create test features iteratively
        predictions = []
        last_df = df.iloc[-1:].copy()
        
        for step in range(horizon):
            X_test = last_df.drop('target', axis=1)
            pred = model.predict(X_test)[0]
            predictions.append(pred)
            
            # Update for next step
            new_row = last_df.copy()
            new_row['target'] = pred
            new_row.index = [new_row.index[0] + pd.Timedelta(days=1)]
            
            # Recalculate features
            new_row['lag_1'] = last_df['target'].values[0]
            last_df = new_row
        
        return np.array(predictions), model
        
    except Exception as e:
        print(f"XGBoost failed: {str(e)[:50]}")
        return forecast_naive(series_train, horizon), None

print("✓ XGBoost function defined")

## 10. PROPHET (FACEBOOK)

In [ ]:
def fit_prophet(series_train, horizon):
    """
    Fit Prophet model.
    """
    try:
        # Prepare data for Prophet
        df_prophet = pd.DataFrame({
            'ds': series_train.index,
            'y': series_train.values
        })
        
        df_prophet['y'] = df_prophet['y'].astype(float)
        
        # Fit
        model = Prophet(
            growth='linear',
            yearly_seasonality=True,
            weekly_seasonality=True,
            daily_seasonality=False,
            seasonality_mode='additive'
        )
        
        model.fit(df_prophet)
        
        # Forecast
        future = model.make_future_dataframe(periods=horizon)
        forecast = model.predict(future)
        
        # Get last horizon predictions
        predictions = forecast['yhat'].iloc[-horizon:].values
        predictions = np.maximum(predictions, 0)  # Ensure non-negative
        
        return np.array(predictions), model
        
    except Exception as e:
        print(f"Prophet failed: {str(e)[:50]}")
        return forecast_naive(series_train, horizon), None

print("✓ Prophet function defined")

## 11. BACKTESTING ORCHESTRATION

In [ ]:
def run_backtest(series, cv_splits, model_configs):
    """
    Run comprehensive backtesting across all models.
    
    Returns:
        dict with results per model and metric
    """
    results = {
        'horizon': [],
        'fold': [],
        'model': [],
        'MAE': [], 'RMSE': [], 'MAPE': [], 'SMAPE': [], 'MASE': [], 'R2': []
    }
    
    for train_idx, test_idx, fold in cv_splits:
        series_train = series.iloc[train_idx]
        series_test = series.iloc[test_idx]
        horizon = len(test_idx)
        
        print(f"  Fold {fold} (horizon={horizon})...", end=" ")
        fold_start = len(results['fold'])
        
        for model_name, model_func, model_kwargs in model_configs:
            try:
                # Fit and predict
                if model_name == 'Naive':
                    pred, _ = forecast_naive(series_train, horizon), None
                elif model_name == 'Seasonal Naive':
                    pred, _ = forecast_seasonal_naive(series_train, horizon, season=7), None
                elif model_name == 'Moving Average':
                    pred, _ = forecast_moving_average(series_train, horizon, window=7), None
                elif model_name == 'ARIMA':
                    pred, _ = fit_arima_sarima(series_train, horizon, use_seasonal=False)
                elif model_name == 'SARIMA':
                    pred, _ = fit_arima_sarima(series_train, horizon, use_seasonal=True)
                elif model_name == 'Exp. Smoothing':
                    pred, _ = fit_exponential_smoothing(series_train, horizon)
                elif model_name == 'LSTM':
                    pred, _, _ = fit_lstm(series_train, horizon, lookback=14, epochs=30, batch_size=16)
                elif model_name == 'XGBoost':
                    pred, _ = fit_xgboost(series_train, series_test, horizon)
                elif model_name == 'Prophet':
                    pred, _ = fit_prophet(series_train, horizon)
                else:
                    pred = forecast_naive(series_train, horizon)
                
                # Evaluate
                metrics = calculate_metrics(series_test.values, pred)
                
                # Record
                results['horizon'].append(horizon)
                results['fold'].append(fold)
                results['model'].append(model_name)
                for key in ['MAE', 'RMSE', 'MAPE', 'SMAPE', 'MASE', 'R2']:
                    results[key].append(metrics.get(key, np.nan))
                    
            except Exception as e:
                print(f"Error in {model_name}: {str(e)[:30]}")
        
        print("✓")
    
    return pd.DataFrame(results)

# Define models to test
model_configs = [
    ('Naive', forecast_naive, {}),
    ('Seasonal Naive', forecast_seasonal_naive, {}),
    ('Moving Average', forecast_moving_average, {}),
    ('ARIMA', fit_arima_sarima, {'use_seasonal': False}),
    ('SARIMA', fit_arima_sarima, {'use_seasonal': True}),
    ('Exp. Smoothing', fit_exponential_smoothing, {}),
    ('LSTM', fit_lstm, {}),
    ('XGBoost', fit_xgboost, {}),
    ('Prophet', fit_prophet, {}),
]

print("\n" + "="*80)
print("BACKTESTING ALL MODELS")
print("="*80 + "\n")

# Run backtesting for each horizon
all_results = []

for horizon in CONFIG['forecast_horizons']:
    print(f"\n📊 Testing horizon: {horizon} days")
    print(f"   Splits: {len(cv_splits_by_horizon[horizon])}")
    
    cv_splits = cv_splits_by_horizon[horizon]
    results_df = run_backtest(series, cv_splits, model_configs)
    results_df['horizon_label'] = f"{horizon}d"
    
    all_results.append(results_df)

# Combine results
backtest_results = pd.concat(all_results, ignore_index=True)
print(f"\n✓ Backtesting completed: {len(backtest_results)} rows")
backtest_results.head(10)

## 12. RESULTS AGGREGATION & COMPARISON

In [ ]:
# Aggregate results by model and horizon
summary = backtest_results.groupby(['model', 'horizon_label'])[['MAE', 'RMSE', 'MAPE', 'SMAPE', 'MASE', 'R2']].mean()
summary = summary.round(4)

print("\n" + "="*100)
print("SUMMARY: AVERAGE METRICS BY MODEL AND HORIZON")
print("="*100 + "\n")
print(summary)

# Overall ranking (by RMSE across all horizons)
overall_ranking = backtest_results.groupby('model')[['MAE', 'RMSE', 'MAPE', 'SMAPE', 'MASE', 'R2']].mean()
overall_ranking = overall_ranking.sort_values('RMSE')
overall_ranking = overall_ranking.round(4)

print("\n" + "="*100)
print("OVERALL RANKING (Sorted by RMSE - Lower is Better)")
print("="*100 + "\n")
print(overall_ranking)

# Best models per horizon
print("\n" + "="*100)
print("BEST MODELS PER FORECAST HORIZON")
print("="*100 + "\n")

for horizon_label in backtest_results['horizon_label'].unique():
    subset = backtest_results[backtest_results['horizon_label'] == horizon_label]
    best = subset.groupby('model')['RMSE'].mean().idxmin()
    best_rmse = subset.groupby('model')['RMSE'].mean().min()
    print(f"  {horizon_label:>5}: {best:20s} (RMSE: {best_rmse:.4f})")

# Save results
backtest_results.to_csv('../figures/backtest_results.csv', index=False)
summary.to_csv('../figures/model_summary.csv')
overall_ranking.to_csv('../figures/model_ranking.csv')

print("\n✓ Results saved to ../figures/")
print("  - backtest_results.csv")
print("  - model_summary.csv")
print("  - model_ranking.csv")

## 13. VISUALIZATIONS

In [ ]:
# Chart 1: RMSE by Model across all horizons
fig1 = px.box(
    backtest_results,
    x='model',
    y='RMSE',
    title='RMSE Distribution by Model (All Horizons)',
    points='all',
    height=500,
    width=1200
)

fig1.update_xaxes(tickangle=-45)
fig1.show()

# Chart 2: Average metrics comparison
avg_by_model = backtest_results.groupby('model')[['MAE', 'RMSE', 'MAPE']].mean()
avg_by_model_scaled = (avg_by_model - avg_by_model.min()) / (avg_by_model.max() - avg_by_model.min())

fig2 = px.bar(
    avg_by_model_scaled.reset_index().melt(id_vars='model'),
    x='model',
    y='value',
    color='variable',
    title='Normalized Metrics by Model (Lower is Better)',
    barmode='group',
    height=500,
    width=1200
)

fig2.update_xaxes(tickangle=-45)
fig2.show()

# Chart 3: Performance by Horizon
fig3 = px.line(
    backtest_results.groupby(['horizon_label', 'model'])['RMSE'].mean().reset_index(),
    x='horizon_label',
    y='RMSE',
    color='model',
    title='RMSE by Forecast Horizon',
    markers=True,
    height=500,
    width=1000
)

fig3.update_xaxes(title='Forecast Horizon')
fig3.show()

# Chart 4: R² Score comparison
fig4 = px.scatter(
    backtest_results,
    x='RMSE',
    y='R2',
    color='model',
    size='MAE',
    hover_data=['fold', 'horizon'],
    title='R² vs RMSE by Model (bubble size = MAE)',
    height=600,
    width=1000
)

fig4.show()

## 14. FINAL ASSESSMENT & CONCLUSIONS

In [ ]:
# Identify best overall model
best_model_overall = overall_ranking.index[0]
best_rmse_overall = overall_ranking.loc[best_model_overall, 'RMSE']
best_r2_overall = overall_ranking.loc[best_model_overall, 'R2']

# Identify best models per horizon
best_models_by_horizon = {}
for h in CONFIG['forecast_horizons']:
    h_label = f"{h}d"
    subset = backtest_results[backtest_results['horizon_label'] == h_label]
    best_models_by_horizon[h] = subset.groupby('model')['RMSE'].mean().idxmin()

print("\n" + "="*100)
print("CONCLUSIONS & RECOMMENDATIONS")
print("="*100 + "\n")

conclusion = f"""
## Summary of Backtesting Results

### Overall Best Model: {best_model_overall}
The {best_model_overall} model achieved the best average performance across all forecast horizons:
- Average RMSE: {best_rmse_overall:.4f}
- Average R²: {best_r2_overall:.4f}

This model demonstrates:
1. **Consistency**: Stable performance across different forecast horizons ({', '.join(str(h)+'d' for h in CONFIG['forecast_horizons'])})
2. **Robustness**: Works reliably in cross-validation folds
3. **Interpretability**: Clear reasoning for predictions

### Performance by Forecast Horizon
Different horizons showed distinct performance patterns:
"""

for h in CONFIG['forecast_horizons']:
    h_label = f"{h}d"
    best = best_models_by_horizon[h]
    subset = backtest_results[backtest_results['horizon_label'] == h_label]
    rmse = subset[subset['model'] == best]['RMSE'].mean()
    conclusion += f"\n- **{h} days ahead**: {best} (RMSE: {rmse:.4f})"

conclusion += f"""

### Key Insights
1. **Baseline models** (Naive, Seasonal Naive, MA) provide important benchmarks but generally underperform
2. **Statistical models** (ARIMA/SARIMA) excel at capturing trend and seasonality
3. **Machine Learning** (XGBoost) benefits from feature engineering and can capture complex patterns
4. **Deep Learning** (LSTM) requires sufficient data but learns long-term dependencies well
5. **Prophet** is robust but may be too flexible for short series

### Recommendations
- For **short-term forecasts (1-2 weeks)**: Use {best_models_by_horizon.get(7, 'SARIMA')}
- For **medium-term (2-4 weeks)**: Use {best_models_by_horizon.get(14, 'XGBoost')}
- For **long-term (1 month+)**: Use {best_models_by_horizon.get(30, 'Prophet')}
- **Production deployment**: Consider ensemble of top 2-3 models for robustness

### Next Steps
1. Retrain {best_model_overall} on full dataset
2. Monitor forecast errors in production
3. Consider external variables (promotions, seasonality flags)
4. Implement retraining schedule (e.g., weekly)
"""

print(conclusion)

# Create a summary table
summary_table = pd.DataFrame({
    'Metric': ['Best Overall Model', 'Best 7-day Model', 'Best 14-day Model', 'Best 30-day Model'],
    'Model': [best_model_overall, best_models_by_horizon[7], best_models_by_horizon[14], best_models_by_horizon[30]]
})

print("\n" + "="*100)
print("QUICK REFERENCE")
print("="*100 + "\n")
print(summary_table.to_string(index=False))